# Bonus: XCiT versus 1-D EfficientNet

Two models can look at the same IQ samples and still behave very differently. This five-minute comparison puts the released XCiT checkpoint beside the EfficientNet-B0 checkpoint trained in `tutorial.ipynb`. We'll compare model size, inference time, predictions, confidence, and agreement on one shared batch.

> This is a demonstration, not a fair architecture benchmark. XCiT was trained broadly on 57 historical classes, while our EfficientNet checkpoint came from a tiny two-signal tutorial. Training data and objectives matter at least as much as architecture.

## 1. Bring the two checkpoints together

The released XCiT checkpoint downloads automatically. EfficientNet is the model you created in the main tutorial, so this notebook looks beneath `runs/v1_iq_tutorial/checkpoints`. Run `tutorial.ipynb` first, or upload that `.ckpt` file when Colab prompts you.

In [ ]:
import importlib.util
import subprocess
import sys

requirements = []
if importlib.util.find_spec('torchsig_models') is None:
    requirements.append('git+https://github.com/TorchDSP/torchsig-models.git@v1.0.0')
if importlib.util.find_spec('matplotlib') is None:
    requirements.append('matplotlib>=3.7')
if requirements:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *requirements])
    print('Installation complete. Restart the runtime if an import still fails.')
else:
    print('All dependencies are ready.')

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

IN_COLAB = 'google.colab' in sys.modules
HERE = Path.cwd()
if not (HERE / 'runs').exists() and (HERE / 'models_tutorial').exists():
    HERE = HERE / 'models_tutorial'

XCIT_CHECKPOINT = HERE / 'xcit_narrowband_v1.0.0.ckpt'
XCIT_URL = (
    'https://github.com/TorchDSP/torchsig-models/releases/download/'
    'v1.0.0/xcit_narrowband_v1.0.0.ckpt'
)
if not XCIT_CHECKPOINT.exists():
    print('Downloading the official XCiT checkpoint...')
    urlretrieve(XCIT_URL, XCIT_CHECKPOINT)

efficientnet_candidates = sorted(
    (HERE / 'runs/v1_iq_tutorial/checkpoints').glob('best-*.ckpt')
)
if not efficientnet_candidates and IN_COLAB:
    from google.colab import files
    print('Upload the EfficientNet .ckpt produced by tutorial.ipynb.')
    uploaded = files.upload()
    efficientnet_candidates = [Path(name) for name in uploaded if name.endswith('.ckpt')]
if not efficientnet_candidates:
    raise FileNotFoundError(
        'No EfficientNet checkpoint found. Run tutorial.ipynb first or upload its .ckpt file.'
    )
EFF_CHECKPOINT = efficientnet_candidates[-1]
print('XCiT:', XCIT_CHECKPOINT)
print('EfficientNet:', EFF_CHECKPOINT)

## 2. Give both models exactly the same batch

We'll synthesize eight tones and eight rising linear chirps. Frequency, chirp endpoints, phase, and SNR vary, but every example has 4,096 complex samples. This is intentionally lightweight and deterministic—not a substitute for the static TorchSig test set used in the main tutorial.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import torch

from torchsig.signals.signal_types import Signal
from torchsig.transforms.transforms import AWGN
from torchsig_models.models import XCiTClassifier
from torchsig_models.models.iq_models.efficientnet import efficientnet_b0

SEED = 2026
NUM_SAMPLES = 4096
EXAMPLES_PER_CLASS = 8
rng = np.random.default_rng(SEED)
n = np.arange(NUM_SAMPLES)

def add_noise(iq, snr_db):
    # The analytic signals have unit power, so AWGN power in dB is -SNR.
    transform_seed = int(rng.integers(0, 2**32 - 1))
    return AWGN(
        noise_power_db=-float(snr_db), seed=transform_seed
    )(Signal(data=np.asarray(iq, dtype=np.complex64))).data

signals, true_names = [], []
for _ in range(EXAMPLES_PER_CLASS):
    frequency = rng.uniform(-0.30, 0.30)
    phase = rng.uniform(0, 2 * np.pi)
    tone = np.exp(1j * (2 * np.pi * frequency * n + phase))
    signals.append(add_noise(tone, rng.uniform(10, 30)))
    true_names.append('tone')
for _ in range(EXAMPLES_PER_CLASS):
    start_frequency = rng.uniform(-0.35, -0.10)
    end_frequency = rng.uniform(0.10, 0.35)
    slope = (end_frequency - start_frequency) / (NUM_SAMPLES - 1)
    chirp_phase = 2 * np.pi * (start_frequency * n + 0.5 * slope * n**2)
    chirp = np.exp(1j * (chirp_phase + rng.uniform(0, 2 * np.pi)))
    signals.append(add_noise(chirp, rng.uniform(10, 30)))
    true_names.append('lfm-radar')

iq_batch = np.asarray(signals, dtype=np.complex64)
model_input = torch.from_numpy(
    np.stack((iq_batch.real, iq_batch.imag), axis=1)
).float()
print('Shared input:', tuple(model_input.shape))

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 5), constrained_layout=True)
example_indices = [0, 1, 2, 3, 8, 9, 10, 11]
for axis, index in zip(axes.flat, example_indices):
    axis.specgram(iq_batch[index], NFFT=256, Fs=1.0, noverlap=192, cmap='magma')
    axis.set_title(f'{index}: {true_names[index]}')
fig.suptitle('A few examples from the shared comparison batch')
plt.show()

## 3. Restore both architectures

Lightning checkpoints prefix the wrapped EfficientNet weights with `model.`. We remove that one wrapper prefix, infer the classifier width from the saved tensor, and then load the exact state dictionary. XCiT can reconstruct itself directly from the hyperparameters stored in its checkpoint.

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

xcit = XCiTClassifier.load_from_checkpoint(XCIT_CHECKPOINT, map_location=DEVICE)
xcit.to(DEVICE).eval()

eff_payload = torch.load(EFF_CHECKPOINT, map_location='cpu', weights_only=False)
eff_state = eff_payload.get('state_dict', eff_payload)
eff_state = {key.removeprefix('model.'): value for key, value in eff_state.items()}
classifier_weights = [
    value for key, value in eff_state.items() if key.endswith('classifier.weight')
]
if len(classifier_weights) != 1:
    raise ValueError('Could not infer the EfficientNet classifier width.')
eff_num_classes = int(classifier_weights[0].shape[0])
efficientnet = efficientnet_b0(
    num_classes=eff_num_classes, drop_path_rate=0.0, drop_rate=0.0
)
efficientnet.load_state_dict(eff_state, strict=True)
efficientnet.to(DEVICE).eval()

def parameter_count(model):
    return sum(parameter.numel() for parameter in model.parameters())

print(f'Device: {DEVICE}')
print(f'XCiT parameters:         {parameter_count(xcit):,}')
print(f'EfficientNet parameters: {parameter_count(efficientnet):,}')
print(f'EfficientNet outputs:    {eff_num_classes}')

## 4. Measure predictions and latency

Timing GPU work requires synchronization because kernels run asynchronously. We warm up each model, time several complete batch passes, and report milliseconds per batch and per example. This is a quick measurement on the current runtime—not a hardware-independent benchmark.

In [ ]:
CLASS_NAMES_V211 = [
    'tone', 'ofdm-64', 'ofdm-72', 'ofdm-128', 'ofdm-180', 'ofdm-256',
    'ofdm-300', 'ofdm-512', 'ofdm-600', 'ofdm-900', 'ofdm-1024',
    'ofdm-1200', 'ofdm-2048', 'lfm-data', 'lfm-radar', '2fsk', '4fsk',
    '8fsk', '16fsk', '2gfsk', '4gfsk', '8gfsk', '16gfsk', '2msk',
    '4msk', '8msk', '16msk', '2gmsk', '4gmsk', '8gmsk', '16gmsk',
    'fm', 'ook', 'bpsk', 'qpsk', '8psk', '16psk', '32psk', '64psk',
    '4ask', '8ask', '16ask', '32ask', '64ask', '16qam', '32qam',
    '64qam', '256qam', '1024qam', '32qam_cross', '128qam_cross',
    '512qam_cross', 'chirpss', 'am-dsb', 'am-dsb-sc', 'am-usb', 'am-lsb',
]
EFF_TUTORIAL_NAMES = ['tone', 'lfm-radar']
batch = model_input.to(DEVICE)

def synchronize():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()

def predict_and_time(model, repeats):
    with torch.inference_mode():
        for _ in range(2):
            model(batch)
        synchronize()
        started = time.perf_counter()
        for _ in range(repeats):
            logits = model(batch)
        synchronize()
    milliseconds = (time.perf_counter() - started) * 1000 / repeats
    return logits.softmax(dim=1).cpu(), milliseconds

repeats = 20 if DEVICE.type == 'cuda' else 5
xcit_probabilities, xcit_ms = predict_and_time(xcit, repeats)
eff_probabilities, eff_ms = predict_and_time(efficientnet, repeats)

xcit_confidence, xcit_index = xcit_probabilities.max(dim=1)
eff_confidence, eff_index = eff_probabilities.max(dim=1)
xcit_names = [CLASS_NAMES_V211[index] for index in xcit_index.tolist()]
eff_names = [
    EFF_TUTORIAL_NAMES[index] if index < len(EFF_TUTORIAL_NAMES) else f'class-{index}'
    for index in eff_index.tolist()
]
print(f'XCiT:         {xcit_ms:.2f} ms/batch, {xcit_ms / len(batch):.3f} ms/example')
print(f'EfficientNet: {eff_ms:.2f} ms/batch, {eff_ms / len(batch):.3f} ms/example')

In [ ]:
x = np.arange(len(true_names))
width = 0.38
fig, axis = plt.subplots(figsize=(14, 5), constrained_layout=True)
axis.bar(x - width / 2, xcit_confidence.numpy(), width, label='XCiT')
axis.bar(x + width / 2, eff_confidence.numpy(), width, label='EfficientNet')
axis.axvline(EXAMPLES_PER_CLASS - 0.5, color='black', linestyle='--', alpha=0.5)
axis.set(
    title='Top-class confidence on the same IQ examples',
    xlabel='Example (tones on left, chirps on right)',
    ylabel='Softmax confidence', ylim=(0, 1.05), xticks=x,
)
axis.legend()
axis.grid(axis='y', alpha=0.25)
plt.show()

print(f"{'#':>2}  {'truth':11s} {'XCiT':16s} {'EfficientNet':16s}")
for index, (truth, x_name, x_score, e_name, e_score) in enumerate(
    zip(true_names, xcit_names, xcit_confidence, eff_names, eff_confidence)
):
    print(f'{index:2d}  {truth:11s} {x_name:12s} {x_score.item():5.1%}  {e_name:12s} {e_score.item():5.1%}')

## 5. Summarize without overclaiming

Name-level agreement is useful here because the models have different output widths and label histories. Accuracy uses our analytic labels, which are only an approximate stand-in for the TorchSig generators. Read these numbers as a debugging comparison, not a leaderboard.

In [ ]:
truth = np.asarray(true_names)
xcit_array = np.asarray(xcit_names)
eff_array = np.asarray(eff_names)
summary = {
    'XCiT accuracy': np.mean(xcit_array == truth),
    'EfficientNet accuracy': np.mean(eff_array == truth),
    'Model agreement': np.mean(xcit_array == eff_array),
}
for name, value in summary.items():
    print(f'{name:23s}: {value:.1%}')

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), constrained_layout=True)
axes[0].bar(['XCiT', 'EfficientNet'], [parameter_count(xcit), parameter_count(efficientnet)], color=['#4472C4', '#ED7D31'])
axes[0].set(title='Trainable parameters', ylabel='Parameters')
axes[0].ticklabel_format(axis='y', style='plain')
axes[1].bar(['XCiT', 'EfficientNet'], [xcit_ms, eff_ms], color=['#4472C4', '#ED7D31'])
axes[1].set(title=f'Batch latency ({DEVICE.type})', ylabel='Milliseconds')
plt.show()

## Try it during the break

Change the batch before drawing conclusions. Lower the SNR, reverse the chirps, shorten their duration, or add a frequency offset. Then ask three separate questions: which model is faster here, which model is more accurate on this distribution, and where do their mistakes disagree?

For a fair architecture study, retrain both models on identical splits, preprocessing, label vocabularies, epoch budgets, and selection rules. This notebook is the quick diagnostic that helps you design that experiment.